# 5. Visualizing Clusters

*Goal: Help humans understand the multi-dimensional embeddings and their groupings.*

Machine learning models can easily calculate distances across thousands of dimensions, but human brains can only visualize three. In this notebook, we will mathematically compress our high-dimensional embeddings down to 3D coordinates and plot them interactively to get a visualization of similarity between commands.

## Step 1 — Import Libraries

Import the required modules:
- `pandas` for DataFrame operations (I like to import `pandas` as `pd` for short hand)
- `numpy` for working with arrays of embedding vectors (I like to import `numpy` as `np` for short hand)
- `plotly.express` for interactive 3-D scatter plots (I like to import  `plotly.express` as `px` for short hand)
- `PCA` from `sklearn.decomposition` to reduce high-dimensional embeddings to 3 dimensions
- `List` and `Tuple` from `typing` to annotate function signatures

In [ ]:
import pandas as pd  # <--- DataFrame usage for dataset operations
import numpy as np  # <--- Numpy arrays for clustering algorithms
import plotly.express as px  # <--- Fancy graphing!
from sklearn.decomposition import PCA  # <--- Principal Component Analysis for graphing
from typing import List, Tuple  # <--- Typing to help convey variable types

## Step 2 — Load the Dataset

Let's pick up right where we left off. We need to load the Parquet file we generated at the end of the last notebook.

Use `pd.read_parquet()` to load `"_04_clustered_commands.parquet"` into a variable called `dataframe`.

 > *Reminder:* This file contains the `CommandLine`, `Embedding`, `Cluster_DBSCAN`, and `Cluster_KMeans` columns needed for our visualization.

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html

In [ ]:
dataframe = pd.read_parquet("_04_clustered_commands.parquet")

## Step 3 — Deduplicate to Unique Commands

We need to make a unique mapping of commands so we will use the DataFrame function `drop_duplicates(subset="CommandLine")` to produce one row per unique command, then reset the index (do this by calling the `.reset_index(drop=True)` function). Assign the new unique DataFrame to a variabled called `uninque_df`.

Example:
```python
dataframe.drop_duplicates(subset=<COLUMN TO DEDUPLICATE BY>)\
    .reset_index(drop=True)
```

> Risk analysis is performed per cluster of unique commands — not per raw event. Deduplicating here avoids sending the same command multiple times in the prompt.

Related Docs:
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html

In [ ]:
# Drop duplicate CommandLines so we have a unique list of CommandLines
# and their respected embeddings/vectors
unique_df = dataframe.drop_duplicates(subset="CommandLine")\
    .reset_index(drop=True)

Create a variable `unique_command_lines` that is a numpy array of unique commands by calling the `CommandLine` column / series function `.to_numpy()`.

Example:
```python
unique_df[<COLUMN>].to_numpy()
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.to_numpy.html

In [ ]:
unique_command_lines = unique_df["CommandLine"].to_numpy()

Now we need to create an array of the embeddings and assign it to the `unique_command_lines_vectors` variable. We can use the numpy function `stack()` and pass it the `Embedding` column / series.

Example:
```python
np.stack(unique_df[<COLUMN>])
```

In [ ]:
unique_command_lines_vectors = np.stack(unique_df["Embedding"])

## Step 4 — Reduce Dimensions with PCA

Your embedding vectors currently have hundreds or thousands of dimensions (e.g., 3,072 if using OpenAI). We cannot plot 3,072 axes on a screen. 

We will use **Principal Component Analysis (PCA)**. PCA is an algorithm that squishes high-dimensional data down to a lower dimension while preserving the relative distances and variances between the points.

First, create a PCA class and assign it to a variable called `pca`. Pass it the number of dimentions we want to produce. We will also pass it the `n_oversamples=1`.

Example:
```python
PCA(3, n_oversamples=1)
```

Related Docs:
 - https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

In [ ]:
pca = PCA(3, n_oversamples=1)

Now call the `pca.fit()` function and pass it the `unique_command_lines_vectors` variable.

In [ ]:
pca.fit(unique_command_lines_vectors)

Once the data has been fit, we can create an array of three dimentional data by calling `pca.transform()` and passing it the `unique_command_lines_vectors` variable. Assign the results to a `three_dimensions` variable.

Example:
```python
pca.transform(unique_command_lines_vectors)
```

In [ ]:
three_dimensions = pca.transform(unique_command_lines_vectors)

`unique_command_lines_vectors` and `three_dimensions` arrays correlate with eachother. You can look at the 3 dimensional version of a vector like this:

```python
print("Example: {} => {}".format(unique_command_lines_vectors[0], three_dimensions[0]))
```

In [ ]:
print("Example: {} => {}".format(unique_command_lines_vectors[0], three_dimensions[0]))

## Step 5 — Define the 3-D Scatter Plot Helper

Define a reusable `scatter3d()` function that:
1. Builds a DataFrame of `x`, `y`, `z` coordinates paired with their cluster label
2. Optionally filters out noise points (cluster `-1`) when `exclude_unclustered=True`
3. Renders an interactive 3-D scatter plot coloured by cluster using Plotly Express
4. Saves the figure to `plot.html` and returns it for inline display

> This function is called once for DBSCAN results and once for KMeans results in the steps below.

In [ ]:
# Function that will create a scatter plot figure of vectors in 3 dimensions
def scatter3d(
        data: List[Tuple[float, float, float]], 
        labels: List[int], 
        exclude_unclustered=False
    ):
    """Data is a list of 3d vectors. Labels are the clusters that correlate to the a given vector.
    """
    # Create a DataFrame that has the X, Y, Z coordinates of each item.
    _df = pd.DataFrame({
        "cluster": labels,
        "x": data[:, 0],
        "y": data[:, 1],
        "z": data[:, 2]
    })
    # Exclude unclusted data if requested
    if exclude_unclustered:
        _df = _df[_df["cluster"] != -1]

    # Create a figure with our DataFrame
    _fig = px.scatter_3d(
        _df,
        x='x', y='y', z='z',
        color='cluster'
    )
    _fig.write_html("plot.html")

    # Return the figure
    return _fig

## Step 6 — Visualise DBSCAN Clusters
Create a variable called `labels` and assign it the `Cluster_DBSCAN` column of the unique dataframe (`unique_df`)

In [ ]:
labels = unique_df["Cluster_DBSCAN"]

Call the `scatter3d()` function and pass it the `three_dimensions`, `labels` parameters and set `exclude_unclustered=True`.

Example:
```python
scatter3d(three_dimensions, labels, exclude_unclustered=True)
```

> Rotate and zoom the chart to explore how well DBSCAN separated the command space. Tightly packed, well-separated blobs of the same colour indicate strong clusters.

In [ ]:
scatter3d(three_dimensions, labels, exclude_unclustered=True)

## Step 7 — Inspect a Specific DBSCAN Cluster

Remember, you can filter `unique_df` to a single DBSCAN cluster number and display the first few rows to read the actual command lines that belong to it.

Example:
```python
unique_df[unique_df["Cluster_DBSCAN"]==29]
```

> Change the cluster number to explore different groups. Look for commands that share a common tool, technique, or attacker pattern.

In [ ]:
# View an example of a cluster
unique_df[unique_df["Cluster_DBSCAN"]==29]

## Step 8 — Visualise KMeans Clusters

Let's compare our DBSCAN perspective against our KMeans perspective. Remember, KMeans forced *every single point* into a cluster.

 1. Update your `labels` variable to equal the `"Cluster_KMeans"` column from `unique_df`.
 2. Call the `scatter3d()` function again, passing in `three_dimensions`, `labels`, and `exclude_unclustered=True`.

In [ ]:
# Plot the data
labels = unique_df["Cluster_KMeans"]
scatter3d(three_dimensions, labels, exclude_unclustered=True)

## Step 9 — Inspect a Specific KMeans Cluster

Let's spot-check a KMeans cluster to see if the semantic meaning held up despite KMeans forcing outliers into the groupings.

 1. Pick a cluster number from your new KMeans graph.
 2. Filter your `unique_df` where `"Cluster_KMeans"` equals that number.
 3. View the first few rows using `.head(10)`.

In [ ]:
# View an example of a cluster
unique_df[unique_df["Cluster_KMeans"]==13].head()